# Predicting Airbnb Occupancy and Pricing Using Listing and Neighborhood Demographic Data

**Authors:** Roman Shrestha and Tanish Pradhan Wong Ah Sui
**Course:** CS 395 — Machine Learning
**Repo:** https://github.com/tanwasp/airbnb-equity-ml

This notebook is the analysis companion to our written rough draft. It runs end-to-end
from the cleaned data files in `data/` to the figures and tables in the report.

## 1. Introduction & motivation

The short term rental market, disrupted by Airbnb in the early 2010s, has grown into a major
economic force in cities across the world. Behind that marketplace sits a layer of algorithmic
systems that decide which listings get prominence in search results and how prices are recommended.
These systems often lack transparency, and prior work in algorithmic fairness has consistently shown
that opaque platform systems can reinforce existing patterns of inequality.

There is a small but growing body of research on Airbnb specifically. Edelman, Luca, and Svirsky (2017)
ran a field experiment and found that guest applications with distinctively African American names were
16% less likely to be accepted than identical applications from guests with distinctively white names.
Tornberg (2022) examined more than 834,000 listings across 97 cities, and found that Black hosts received
on average 22% less revenue than comparable white hosts. Together these papers show that Airbnb activity
is not evenly distributed across urban neighborhoods and may reflect existing patterns of economic
segregation.

Our personal motivation comes from being international students from the Global South. After moving to
the US for school we saw a version of the same dynamics we grew up around (gentrification, uneven growth)
but increasingly mediated by digital platforms that operate with little transparency. This project is
our attempt to look at one of those platforms with the tools we have learned in this class.

## 2. Research questions

1. Can machine learning models accurately predict which Airbnb listings result in higher occupancy,
   and which listing factors are most predictive?
2. What listing and host characteristics are most strongly associated with Airbnb pricing in the US?
3. Do neighborhood level demographics (household income, poverty rate, racial composition)
   significantly predict occupancy and prices, independent of or in addition to listing level
   attributes?

Following Prof. Stapleton's feedback on our pitch and proposal, we also pushed further on the equity
angle. Specifically we wanted to know not just whether demographics show up as predictors, but whether
the resulting model behaves differently across protected groups.

## 3. Setup

All of the imports we need across the notebook. We pin `random_state=395` everywhere for
reproducibility.

In [ ]:
# the notebook lives in progress_report/ but the data is at project_root/data/.
# this makes the notebook work whether you run it from progress_report/ or from
# the project root (e.g. in jupyter classic vs vscode).
import os
if not os.path.exists("data/airbnb_cleaned_part1.csv"):
    os.chdir("..")
print("working directory:", os.getcwd())

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                             accuracy_score, classification_report, roc_auc_score)

import xgboost as xgb
import lightgbm as lgb
import shap

SEED = 395
OUT = "progress_report"

## 4. Data description and loading

Our primary data source is **Inside Airbnb** (insideairbnb.com), an open source project that publishes
Airbnb listing data scraped for major cities around the world. We downloaded the 2025 listing files for
34 US cities and concatenated them. To get neighborhood demographics, we geocoded each listing to its
2020 census tract and pulled variables from the **2023 American Community Survey 5-year estimates (ACS5)**
via the Census Bureau API, then merged on tract FIPS.

The merged dataset has 280,286 listings and 61 variables. Each row is a unique 2025 listing.
LA, NYC, and Hawaii are the three largest shares (16.0%, 12.9%, and 11.9% respectively).

The cleaned data is stored split across two CSVs because of GitHub's file size limit. We concatenate
them back here.

In [ ]:
df1 = pd.read_csv("data/airbnb_cleaned_part1.csv", index_col=0)
df2 = pd.read_csv("data/airbnb_cleaned_part2.csv", index_col=0)
df = pd.concat([df1, df2], ignore_index=True)
print("shape:", df.shape)

## 5. Cleaning and preparation

We did several rounds of cleaning. The basics: standardizing column names across the city files,
dropping fully empty or near constant columns, and parsing booleans (`host_is_superhost`,
`instant_bookable`, etc.) which were inconsistently encoded as `True`/`False`/`t`/`f`/`0`/`1`
across files.

A few specific gotchas we hit:

- `host_response_rate` and `host_acceptance_rate` come in as strings like `"95%"` — convert to
  numeric proportions.
- The ACS `median_age` column has Census sentinel values (negative numbers used to flag missing) —
  replace with NaN.
- `property_type` has many categories — collapse to top 5 + "other" to avoid one-hot explosion.

Missing data turned out to be the big preparation challenge (covered in section 8). The headline
numbers: **43.1% of listings are missing the price field**, **20.7% are missing review scores**,
and four cities (LA, NYC, San Francisco, Portland) have no price data at all in the files we
downloaded — so the price model effectively excludes those four cities.

In [ ]:
# fix the median_age sentinel
df["median_age"] = df["median_age"].where(df["median_age"] > 0, other=np.nan)

# percent strings -> proportions
for c in ["host_response_rate", "host_acceptance_rate"]:
    df[c] = df[c].astype(str).str.rstrip("%").replace("nan", np.nan)
    df[c] = pd.to_numeric(df[c], errors="coerce") / 100.0

# bool cols come in as a mess - normalize them
bool_cols = ["host_identity_verified", "host_has_profile_pic",
             "instant_bookable", "host_is_superhost"]
for c in bool_cols:
    df[c] = df[c].map({True: 1, False: 0, "True": 1, "False": 0,
                       "t": 1, "f": 0, 1: 1, 0: 0})
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

# group rare property types
top_props = df["property_type"].value_counts().head(5).index
df["property_type_grouped"] = df["property_type"].where(df["property_type"].isin(top_props), "other")
print("property type buckets:", df["property_type_grouped"].value_counts().to_dict())

### Missingness summary

We focus on a handful of columns that matter for modeling. Price/revenue missingness is the big
problem. Review scores are missing for ~21% of listings (mostly listings with zero reviews).

In [ ]:
key_cols = ["price", "estimated_occupancy_l365d", "estimated_revenue_l365d",
            "review_scores_rating", "median_household_income", "poverty_rate",
            "pct_white", "pct_black", "pct_hispanic", "pct_college_educated"]
miss = df[key_cols].isnull().sum()
miss_pct = (miss / len(df) * 100).round(2)
pd.DataFrame({"count": miss, "pct": miss_pct}).sort_values("pct", ascending=False)

### Summary statistics

Numeric summary of the variables we use most. Note that `price` is right-skewed (mean $221, median
$161) and `estimated_occupancy_l365d` ranges 0-255 nights with median 30.

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols].describe().T[["mean", "std", "min", "50%", "max"]].round(2)

## 6. Exploratory data analysis

We do six visualisations: target distributions, a correlation heatmap, room-type comparison,
demographics-vs-price scatter, listings per city, and demographics-vs-occupancy scatter.

### Figure 1 - target distributions

Price is right-skewed with a long tail. Estimated revenue follows a similar shape. Occupancy is
**bimodal** — a big mass near zero (inactive listings) and a smaller cluster at high occupancy
(professionally managed listings). The bimodality is part of why we frame the occupancy task as
a classification problem rather than regression.

In [ ]:
df_price = df.dropna(subset=["price"])
df_price = df_price[df_price["price"] > 0]
print("with valid price:", len(df_price))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df_price["price"], bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("Nightly Price ($)"); axes[0].set_xlabel("Price"); axes[0].set_ylabel("Count")

axes[1].hist(df["estimated_occupancy_l365d"], bins=50, color="steelblue", edgecolor="white")
axes[1].set_title("Estimated Occupancy (last 365 days)"); axes[1].set_xlabel("Nights booked")

df_rev = df.dropna(subset=["estimated_revenue_l365d"])
axes[2].hist(df_rev["estimated_revenue_l365d"], bins=50, color="steelblue", edgecolor="white")
axes[2].set_title("Estimated Revenue (last 365 days)"); axes[2].set_xlabel("Revenue ($)")

plt.tight_layout(); plt.show()

### Figure 2 - correlation heatmap

Some takeaways from the matrix:

- Price correlates with `accommodates` (0.52), `bedrooms` (0.44), and `median_home_value` (0.27).
- Revenue and occupancy have a strong correlation (0.83) by construction.
- `pct_white` and `pct_black` are negatively correlated (-0.52) — they are roughly substitute
  shares of the tract.
- `median_household_income` correlates positively with `pct_white` and `pct_college_educated`.

In [ ]:
corr_cols = ["price", "estimated_occupancy_l365d", "estimated_revenue_l365d",
             "review_scores_rating", "accommodates", "bedrooms", "beds",
             "number_of_reviews", "reviews_per_month",
             "median_household_income", "poverty_rate", "pct_renter_occupied",
             "pct_white", "pct_black", "pct_hispanic", "pct_college_educated",
             "unemployment_rate"]
corr = df[corr_cols].corr()
fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, ax=ax, square=True, annot_kws={"size": 7})
ax.set_title("Correlation Matrix: Listing and Demographic Features")
plt.tight_layout(); plt.show()

### Figure 3 - room type

Entire homes are pricier and have more occupancy than private rooms or shared rooms. Hotel rooms
sit between entire homes and private rooms in price.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
order = df_price["room_type"].value_counts().index.tolist()
sns.boxplot(data=df_price, x="room_type", y="price", order=order, ax=axes[0])
axes[0].set_title("Price by Room Type"); axes[0].set_xlabel(""); axes[0].set_ylabel("Price ($)")

sns.boxplot(data=df, x="room_type", y="estimated_occupancy_l365d", order=order, ax=axes[1])
axes[1].set_title("Occupancy by Room Type"); axes[1].set_xlabel(""); axes[1].set_ylabel("Nights")
plt.tight_layout(); plt.show()

### Figure 4 - demographic features vs price

Higher median household income and college education are associated with higher prices. Poverty rate
shows the opposite pattern. The %Black correlation with price is small (~ -0.06 raw) but visible —
we'll come back to this.

In [ ]:
demo_cols = ["median_household_income", "poverty_rate", "pct_white",
             "pct_black", "pct_college_educated", "unemployment_rate"]
demo_labels = ["Median Household Income", "Poverty Rate", "% White",
               "% Black", "% College Educated", "Unemployment Rate"]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, col, lbl in zip(axes.flatten(), demo_cols, demo_labels):
    samp = df_price.dropna(subset=[col]).sample(min(5000, len(df_price)), random_state=SEED)
    ax.scatter(samp[col], samp["price"], alpha=0.15, s=5, color="steelblue")
    r = df_price[[col, "price"]].dropna().corr().iloc[0, 1]
    ax.set_xlabel(lbl); ax.set_ylabel("Price ($)"); ax.set_title(f"{lbl}\nr = {r:.3f}")
plt.tight_layout(); plt.show()

### Figure 5 - listings by city

LA, NYC, and Hawaii dominate the listings count. As noted above, LA and NYC have *no* valid price
field in the files we downloaded, so the price-side analysis below is missing two of the largest
markets.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
df["city"].value_counts().head(15).plot(kind="barh", color="steelblue", ax=ax)
ax.set_xlabel("Number of Listings"); ax.set_title("Top 15 Cities by Number of Listings")
ax.invert_yaxis()
plt.tight_layout(); plt.show()

### Figure 6 - demographic features vs occupancy

The raw correlations with occupancy are weaker than with price. Visually nothing pops — this is an
early hint that listing-level signals (reviews, host responsiveness) will dominate occupancy
prediction, not demographics.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, col, lbl in zip(axes.flatten(), demo_cols, demo_labels):
    samp = df.dropna(subset=[col]).sample(min(5000, len(df)), random_state=SEED)
    ax.scatter(samp[col], samp["estimated_occupancy_l365d"], alpha=0.15, s=5, color="coral")
    r = df[[col, "estimated_occupancy_l365d"]].dropna().corr().iloc[0, 1]
    ax.set_xlabel(lbl); ax.set_ylabel("Occupancy (nights)"); ax.set_title(f"{lbl}\nr = {r:.3f}")
plt.tight_layout(); plt.show()

## 7. Methods

The project has two prediction tasks.

**Price (regression).** Predict per-night listing price from listing + demographic features. Price is
heavily right-skewed, so we use `log1p(price)` as the target. RMSE is reported in dollar space (after
inverse-transforming).

**Occupancy (classification).** Predict whether a listing is "high occupancy" (above the dataset
median of 30 nights/year). Occupancy is bimodal (a big spike at zero and another cluster up high),
which would give a regression model trouble — a binary threshold is more honest.

For each task we fit a linear baseline (linear / logistic), random forest, XGBoost, and LightGBM. For
price we also do an ablation: listing-only vs listing+demographic features. We use a 80/20 train/test
split with `random_state=395` throughout.

### Feature preparation

We split features into two groups: listing-level (size, amenities, host attributes, review scores)
and demographic (tract-level ACS variables). `room_type` and the grouped `property_type` get
one-hot encoded.

In [ ]:
listing_features = [
    "accommodates", "bedrooms", "beds", "bathrooms",
    "minimum_nights", "maximum_nights",
    "availability_365", "availability_30",
    "number_of_reviews", "reviews_per_month",
    "review_scores_rating", "review_scores_cleanliness",
    "review_scores_location", "review_scores_value",
    "host_response_time", "host_response_rate", "host_acceptance_rate",
    "calculated_host_listings_count",
    "has_wifi", "has_parking", "has_pool", "has_gym",
    "instant_bookable", "host_is_superhost",
    "host_identity_verified", "host_has_profile_pic"
]
demo_features = [
    "median_household_income", "median_gross_rent", "median_home_value",
    "total_population", "median_age", "poverty_rate",
    "pct_renter_occupied", "pct_white", "pct_black",
    "pct_asian", "pct_hispanic", "pct_college_educated",
    "unemployment_rate"
]

room_dummies = pd.get_dummies(df["room_type"], prefix="room", drop_first=True)
prop_dummies = pd.get_dummies(df["property_type_grouped"], prefix="prop", drop_first=True)

feat = pd.concat([df[listing_features + demo_features], room_dummies, prop_dummies], axis=1)
bcols = feat.select_dtypes(include=["bool"]).columns
feat[bcols] = feat[bcols].astype(int)
print("total features:", feat.shape[1])

### 7.1 Price prediction — baselines

For the price baseline we drop rows with any missing feature (we revisit imputation in section 8).
This gives 128,078 listings with no missing values, predicting `log1p(price)`.

In [ ]:
price_mask = df["price"].notna() & (df["price"] > 0)
X = feat[price_mask].copy()
y = np.log1p(df.loc[price_mask, "price"])

ok = X.notna().all(axis=1)
X = X[ok]; y = y[ok]
X = X.fillna(X.median())  # any stragglers
print("samples:", len(X))

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED)
y_te_actual = np.expm1(y_te)

**Linear regression baseline.**

In [ ]:
sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_te_s = sc.transform(X_te)

lr = LinearRegression()
lr.fit(X_tr_s, y_tr)
y_pred_lr = lr.predict(X_te_s)
y_pred_lr_actual = np.expm1(y_pred_lr)

rmse_lr = np.sqrt(mean_squared_error(y_te_actual, y_pred_lr_actual))
mae_lr  = mean_absolute_error(y_te_actual, y_pred_lr_actual)
r2_lr   = r2_score(y_te, y_pred_lr)

print(f"linear: R2={r2_lr:.4f}  RMSE=${rmse_lr:.2f}  MAE=${mae_lr:.2f}")

coef_df = pd.DataFrame({"feature": X_tr.columns, "coefficient": lr.coef_})
coef_df.sort_values("coefficient", key=abs, ascending=False).head(10)

**Random forest.** Substantially better than linear because it handles the non-linear interactions
and right-tail without us having to specify them.

In [ ]:
rf_reg = RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_leaf=10,
                               random_state=SEED, n_jobs=-1)
rf_reg.fit(X_tr, y_tr)
y_pred_rf = rf_reg.predict(X_te)
y_pred_rf_actual = np.expm1(y_pred_rf)

rmse_rf = np.sqrt(mean_squared_error(y_te_actual, y_pred_rf_actual))
mae_rf  = mean_absolute_error(y_te_actual, y_pred_rf_actual)
r2_rf   = r2_score(y_te, y_pred_rf)
print(f"rf: R2={r2_rf:.4f}  RMSE=${rmse_rf:.2f}  MAE=${mae_rf:.2f}")

importance_df = pd.DataFrame({"feature": X_tr.columns,
                              "importance": rf_reg.feature_importances_})
importance_df = importance_df.sort_values("importance", ascending=False)
importance_df.head(15)

### 7.2 Price ablation: listing-only vs listing+demographics

This is the most direct test of our research question 3. Train two RFs, one with only the listing
features, one with everything, and compare R².

In [ ]:
listing_only = [c for c in X_tr.columns if c not in demo_features]
rf_l = RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_leaf=10,
                             random_state=SEED, n_jobs=-1)
rf_l.fit(X_tr[listing_only], y_tr)
y_pred_l = rf_l.predict(X_te[listing_only])
r2_listing = r2_score(y_te, y_pred_l)
rmse_listing = np.sqrt(mean_squared_error(y_te_actual, np.expm1(y_pred_l)))
print(f"listing only: R2={r2_listing:.4f}  RMSE=${rmse_listing:.2f}")
print(f"listing+demo: R2={r2_rf:.4f}  RMSE=${rmse_rf:.2f}")
print(f"delta R2: {r2_rf - r2_listing:+.4f}")

### 7.3 Occupancy classification — baselines

We define `high_occupancy = 1` if `estimated_occupancy_l365d > median (30)`. Roughly a 50/50 split.

In [ ]:
med_occ = df["estimated_occupancy_l365d"].median()
df["high_occupancy"] = (df["estimated_occupancy_l365d"] > med_occ).astype(int)
print(f"median occupancy = {med_occ} nights, "
      f"{df['high_occupancy'].mean()*100:.1f}% above median")

X_o = feat.copy()
y_o = df["high_occupancy"]
ok_o = X_o.notna().all(axis=1) & y_o.notna()
X_o = X_o[ok_o]; y_o = y_o[ok_o]
X_o = X_o.fillna(X_o.median())
print("samples:", len(X_o))

X_tr_o, X_te_o, y_tr_o, y_te_o = train_test_split(
    X_o, y_o, test_size=0.2, random_state=SEED, stratify=y_o)

**Logistic regression baseline.**

In [ ]:
sc_o = StandardScaler()
X_tr_o_s = sc_o.fit_transform(X_tr_o)
X_te_o_s = sc_o.transform(X_te_o)

log_reg = LogisticRegression(max_iter=1000, random_state=SEED)
log_reg.fit(X_tr_o_s, y_tr_o)
y_pred_log = log_reg.predict(X_te_o_s)
y_prob_log = log_reg.predict_proba(X_te_o_s)[:, 1]
acc_log = accuracy_score(y_te_o, y_pred_log)
auc_log = roc_auc_score(y_te_o, y_prob_log)
print(f"logistic: acc={acc_log:.4f}  AUC={auc_log:.4f}")
print(classification_report(y_te_o, y_pred_log, target_names=["Low", "High"]))

**Random forest classifier.**

In [ ]:
rf_clf = RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_leaf=10,
                                random_state=SEED, n_jobs=-1)
rf_clf.fit(X_tr_o, y_tr_o)
y_pred_clf = rf_clf.predict(X_te_o)
y_prob_clf = rf_clf.predict_proba(X_te_o)[:, 1]
acc_rf = accuracy_score(y_te_o, y_pred_clf)
auc_rf = roc_auc_score(y_te_o, y_prob_clf)
print(f"rf: acc={acc_rf:.4f}  AUC={auc_rf:.4f}")

imp_occ_df = pd.DataFrame({"feature": X_tr_o.columns,
                           "importance": rf_clf.feature_importances_})
imp_occ_df = imp_occ_df.sort_values("importance", ascending=False)
imp_occ_df.head(15)

### 7.4 Occupancy ablation: listing-only vs listing+demographics

For occupancy the demographics ablation barely moves AUC at all. That is consistent with what we saw
in EDA - the listing-level review/host signals dominate.

In [ ]:
listing_only_o = [c for c in X_tr_o.columns if c not in demo_features]
rf_clf_l = RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_leaf=10,
                                  random_state=SEED, n_jobs=-1)
rf_clf_l.fit(X_tr_o[listing_only_o], y_tr_o)
y_pred_lo = rf_clf_l.predict(X_te_o[listing_only_o])
y_prob_lo = rf_clf_l.predict_proba(X_te_o[listing_only_o])[:, 1]
acc_listing = accuracy_score(y_te_o, y_pred_lo)
auc_listing = roc_auc_score(y_te_o, y_prob_lo)
print(f"listing only: acc={acc_listing:.4f}  AUC={auc_listing:.4f}")
print(f"listing+demo: acc={acc_rf:.4f}  AUC={auc_rf:.4f}")
print(f"delta acc={acc_rf - acc_listing:+.4f}  delta AUC={auc_rf - auc_listing:+.4f}")

### Comparison plots — baselines

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(["Linear\nRegression", "Random\nForest"], [r2_lr, r2_rf],
            color=["#4A90D9", "#2ECC71"])
axes[0].set_ylabel("R² Score"); axes[0].set_title("Price Prediction: Model Comparison")
axes[0].set_ylim(0, 1)
for i, v in enumerate([r2_lr, r2_rf]):
    axes[0].text(i, v + 0.02, f"{v:.3f}", ha="center", fontweight="bold")

axes[1].bar(["Logistic\nRegression", "Random\nForest"], [auc_log, auc_rf],
            color=["#4A90D9", "#2ECC71"])
axes[1].set_ylabel("AUC Score"); axes[1].set_title("Occupancy Classification: Model Comparison")
axes[1].set_ylim(0, 1)
for i, v in enumerate([auc_log, auc_rf]):
    axes[1].text(i, v + 0.02, f"{v:.3f}", ha="center", fontweight="bold")

plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
t15 = importance_df.head(15)
axes[0].barh(range(len(t15)), t15["importance"].values, color="steelblue")
axes[0].set_yticks(range(len(t15))); axes[0].set_yticklabels(t15["feature"].values)
axes[0].invert_yaxis(); axes[0].set_xlabel("Feature Importance")
axes[0].set_title("Price Prediction: Top 15 Features (RF)")

t15o = imp_occ_df.head(15)
axes[1].barh(range(len(t15o)), t15o["importance"].values, color="coral")
axes[1].set_yticks(range(len(t15o))); axes[1].set_yticklabels(t15o["feature"].values)
axes[1].invert_yaxis(); axes[1].set_xlabel("Feature Importance")
axes[1].set_title("Occupancy Classification: Top 15 Features (RF)")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
labels = ["Listing Only", "Listing +\nDemographics"]
colors = ["#95a5a6", "#2ECC71"]

vals = [r2_listing, r2_rf]
bars = axes[0].bar(labels, vals, color=colors)
axes[0].set_ylabel("R² Score"); axes[0].set_title("Price Prediction: Effect of Demographics")
axes[0].set_ylim(0, max(vals) * 1.2)
for b, v in zip(bars, vals):
    axes[0].text(b.get_x() + b.get_width()/2, v + 0.01, f"{v:.4f}",
                 ha="center", fontweight="bold")

vals = [auc_listing, auc_rf]
bars = axes[1].bar(labels, vals, color=colors)
axes[1].set_ylabel("AUC Score"); axes[1].set_title("Occupancy Classification: Effect of Demographics")
axes[1].set_ylim(0, max(vals) * 1.2)
for b, v in zip(bars, vals):
    axes[1].text(b.get_x() + b.get_width()/2, v + 0.01, f"{v:.4f}",
                 ha="center", fontweight="bold")
plt.tight_layout(); plt.show()

## 8. Handling missing data

Missingness was a real concern from the progress report, so we ran a side-by-side comparison of four
strategies on the price prediction task using a baseline random forest.

| strategy | tradeoff |
|---|---|
| drop rows with any missing feature | best R² but loses ~25% of data |
| mean imputation | preserves all rows, similar R² |
| median imputation | preserves all rows, similar R² (chosen for downstream) |
| KNN imputation (k=5, 30k subsample) | slower, worse R²; high-dimensional mixed-scale feature space hurts |

We chose median imputation as the downstream default because it preserves all rows and is robust
to skew (price and occupancy are both heavily skewed).

In [ ]:
def fit_rf(X, y, label):
    Xt, Xe, yt, ye = train_test_split(X, y, test_size=0.2, random_state=SEED)
    rf = RandomForestRegressor(n_estimators=150, max_depth=15, min_samples_leaf=10,
                               random_state=SEED, n_jobs=-1)
    rf.fit(Xt, yt)
    yp = rf.predict(Xe)
    r2 = r2_score(ye, yp)
    rmse = np.sqrt(mean_squared_error(np.expm1(ye), np.expm1(yp)))
    print(f"  {label}: R2={r2:.4f}  RMSE=${rmse:.2f}  n={len(X)}")
    return r2, rmse

# X_all = feat[price_mask] BEFORE the dropna step from section 7
X_all = feat[price_mask].copy()
y_all = np.log1p(df.loc[price_mask, "price"])

print("[1] drop rows with any missing")
ok = X_all.notna().all(axis=1)
r2_drop, rmse_drop = fit_rf(X_all[ok], y_all[ok], "drop")

print("\n[2] mean imputation")
imp = SimpleImputer(strategy="mean")
X2 = pd.DataFrame(imp.fit_transform(X_all), columns=X_all.columns, index=X_all.index)
r2_mean, rmse_mean = fit_rf(X2, y_all, "mean")

print("\n[3] median imputation")
imp = SimpleImputer(strategy="median")
X3 = pd.DataFrame(imp.fit_transform(X_all), columns=X_all.columns, index=X_all.index)
r2_med, rmse_med = fit_rf(X3, y_all, "median")

print("\n[4] KNN imputation (k=5, 30k subsample)")
samp = X_all.sample(n=min(30000, len(X_all)), random_state=SEED)
samp_y = y_all.loc[samp.index]
knn = KNNImputer(n_neighbors=5)
X4 = pd.DataFrame(knn.fit_transform(samp), columns=samp.columns, index=samp.index)
r2_knn, rmse_knn = fit_rf(X4, samp_y, "knn")

## 9. Advanced modeling — XGBoost, LightGBM, hyperparameter tuning, CV

We use the median-imputed dataset from above for the rest. Tuned XGBoost is our best model on both
tasks — R² 0.807 on price and AUC 0.986 on occupancy.

We did the grid search on a 40k subsample of the training set with 3-fold CV (the full grid on the
full train would take forever) and refit the best config on the full training set.

In [ ]:
# fresh median-imputed datasets for the advanced models
imp_p = SimpleImputer(strategy="median")
X_price_imp = pd.DataFrame(imp_p.fit_transform(X_all), columns=X_all.columns, index=X_all.index)
y_price = y_all

X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_price_imp, y_price, test_size=0.2, random_state=SEED)
y_te2_actual = np.expm1(y_te2)

# xgboost (default-ish)
print("xgboost...")
xgb_reg = xgb.XGBRegressor(n_estimators=300, max_depth=8, learning_rate=0.05,
                           subsample=0.8, colsample_bytree=0.8,
                           random_state=SEED, n_jobs=-1)
xgb_reg.fit(X_tr2, y_tr2)
yp = xgb_reg.predict(X_te2)
r2_xgb_p = r2_score(y_te2, yp)
rmse_xgb_p = np.sqrt(mean_squared_error(y_te2_actual, np.expm1(yp)))
mae_xgb_p = mean_absolute_error(y_te2_actual, np.expm1(yp))
print(f"  R2={r2_xgb_p:.4f}  RMSE=${rmse_xgb_p:.2f}  MAE=${mae_xgb_p:.2f}")

# lightgbm
print("lightgbm...")
lgb_reg = lgb.LGBMRegressor(n_estimators=300, max_depth=8, learning_rate=0.05,
                            num_leaves=63, subsample=0.8, colsample_bytree=0.8,
                            random_state=SEED, n_jobs=-1, verbose=-1)
lgb_reg.fit(X_tr2, y_tr2)
yp = lgb_reg.predict(X_te2)
r2_lgb_p = r2_score(y_te2, yp)
rmse_lgb_p = np.sqrt(mean_squared_error(y_te2_actual, np.expm1(yp)))
mae_lgb_p = mean_absolute_error(y_te2_actual, np.expm1(yp))
print(f"  R2={r2_lgb_p:.4f}  RMSE=${rmse_lgb_p:.2f}  MAE=${mae_lgb_p:.2f}")

### Grid search and refit

In [ ]:
n_tune = min(40000, len(X_tr2))
X_tune = X_tr2.sample(n=n_tune, random_state=SEED)
y_tune = y_tr2.loc[X_tune.index]

param_grid = {
    "max_depth": [6, 8, 10],
    "learning_rate": [0.05, 0.1],
    "n_estimators": [200, 400],
}
xgb_base = xgb.XGBRegressor(subsample=0.8, colsample_bytree=0.8,
                            random_state=SEED, n_jobs=-1)
grid = GridSearchCV(xgb_base, param_grid, cv=3, scoring="r2", n_jobs=2)
grid.fit(X_tune, y_tune)
print("best:", grid.best_params_)
print("CV R2:", round(grid.best_score_, 4))

best_xgb = grid.best_estimator_
best_xgb.fit(X_tr2, y_tr2)
yp = best_xgb.predict(X_te2)
r2_best_p = r2_score(y_te2, yp)
rmse_best_p = np.sqrt(mean_squared_error(y_te2_actual, np.expm1(yp)))
print(f"tuned full-train: R2={r2_best_p:.4f}  RMSE=${rmse_best_p:.2f}")

### 5-fold cross-validation (price)

Tight CV scores - we are not just lucky on one train/test split.

In [ ]:
idx = np.random.RandomState(SEED).choice(len(X_price_imp), size=min(50000, len(X_price_imp)), replace=False)
X_cv = X_price_imp.iloc[idx]
y_cv = y_price.iloc[idx]
cv_scores = cross_val_score(
    xgb.XGBRegressor(**grid.best_params_, subsample=0.8, colsample_bytree=0.8,
                     random_state=SEED, n_jobs=-1),
    X_cv, y_cv, cv=5, scoring="r2", n_jobs=2)
print("fold R2s:", [round(s, 4) for s in cv_scores])
print(f"mean={cv_scores.mean():.4f}  std={cv_scores.std():.4f}")

### Occupancy gradient boosting + CV

In [ ]:
imp_o2 = SimpleImputer(strategy="median")
X_o_imp = pd.DataFrame(imp_o2.fit_transform(X_o), columns=X_o.columns, index=X_o.index)

X_tr_o2, X_te_o2, y_tr_o2, y_te_o2 = train_test_split(
    X_o_imp, y_o, test_size=0.2, random_state=SEED, stratify=y_o)

print("xgboost...")
xgb_clf = xgb.XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.05,
                            subsample=0.8, colsample_bytree=0.8,
                            random_state=SEED, n_jobs=-1, eval_metric="logloss")
xgb_clf.fit(X_tr_o2, y_tr_o2)
yp_o = xgb_clf.predict(X_te_o2)
yprob_o = xgb_clf.predict_proba(X_te_o2)[:, 1]
acc_xgb_o = accuracy_score(y_te_o2, yp_o)
auc_xgb_o = roc_auc_score(y_te_o2, yprob_o)
print(f"  acc={acc_xgb_o:.4f}  AUC={auc_xgb_o:.4f}")

print("lightgbm...")
lgb_clf = lgb.LGBMClassifier(n_estimators=300, max_depth=8, learning_rate=0.05,
                             num_leaves=63, subsample=0.8, colsample_bytree=0.8,
                             random_state=SEED, n_jobs=-1, verbose=-1)
lgb_clf.fit(X_tr_o2, y_tr_o2)
yp_o = lgb_clf.predict(X_te_o2)
yprob_o = lgb_clf.predict_proba(X_te_o2)[:, 1]
acc_lgb_o = accuracy_score(y_te_o2, yp_o)
auc_lgb_o = roc_auc_score(y_te_o2, yprob_o)
print(f"  acc={acc_lgb_o:.4f}  AUC={auc_lgb_o:.4f}")

In [ ]:
idx_o = np.random.RandomState(SEED).choice(len(X_o_imp), size=min(50000, len(X_o_imp)), replace=False)
X_cv_o = X_o_imp.iloc[idx_o]
y_cv_o = y_o.iloc[idx_o]
cv_scores_o = cross_val_score(
    xgb.XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.05,
                      subsample=0.8, colsample_bytree=0.8,
                      random_state=SEED, n_jobs=-1, eval_metric="logloss"),
    X_cv_o, y_cv_o, cv=5, scoring="roc_auc", n_jobs=2)
print("fold AUCs:", [round(s, 4) for s in cv_scores_o])
print(f"mean={cv_scores_o.mean():.4f}  std={cv_scores_o.std():.4f}")

### Summary table

| Task | Model | Metric | Score | RMSE / Acc |
|---|---|---|---|---|
| Price | Linear Regression | R² | 0.62 | $275.79 |
| Price | Random Forest | R² | 0.75 | $95.68 |
| Price | XGBoost (default) | R² | 0.80 | $93.06 |
| Price | LightGBM | R² | 0.78 | $96.27 |
| **Price** | **XGBoost (tuned)** | **R²** | **0.81** | **$91.38** |
| Occupancy | Logistic Regression | AUC | 0.896 | 0.822 |
| Occupancy | Random Forest | AUC | 0.964 | 0.907 |
| **Occupancy** | **XGBoost** | **AUC** | **0.986** | **0.937** |
| Occupancy | LightGBM | AUC | 0.986 | 0.936 |

## 10. Interpretation with SHAP

We use SHAP (Shapley Additive exPlanations) to interpret the tuned XGBoost models. SHAP decomposes
each individual prediction into per-feature contributions, so we get both a global feature ranking
(mean |SHAP| per feature) and per-listing explanations. We compute SHAP on a 2,000-row test sample
because `TreeExplainer` on the full test set is too slow.

In [ ]:
n_shap = 2000
shap_idx = np.random.RandomState(SEED).choice(len(X_te2), size=n_shap, replace=False)
X_shap = X_te2.iloc[shap_idx]

expl = shap.TreeExplainer(best_xgb)
shap_vals = expl.shap_values(X_shap)

shap_imp = pd.DataFrame({
    "feature": X_shap.columns,
    "mean_abs_shap": np.abs(shap_vals).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)
shap_imp.head(15)

### Figure 11 - top features for price (SHAP)

The top features are listing size and capacity (`accommodates`, `bedrooms`, `bathrooms`), but
`median_home_value` (a tract-level demographic) is high in the ranking and `pct_black` is in the
top 8. So demographics aren't dominant, but they are clearly part of what drives price.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
top15 = shap_imp.head(15).iloc[::-1]
ax.barh(top15["feature"], top15["mean_abs_shap"], color="#20808D")
ax.set_xlabel("Mean |SHAP value| (impact on log price)")
ax.set_title("Feature importance for price (XGBoost, SHAP)")
plt.tight_layout(); plt.show()

### Figure 12 - SHAP beeswarm

Each dot is one listing. Color is feature value (red high, blue low). Reading horizontally:
high `accommodates` pushes price up, high `pct_black` pushes price down (the red dots cluster on
the negative side of the axis), high `median_home_value` pushes it up. This is the most direct
visualisation of the demographic effect on price.

In [ ]:
plt.figure()
shap.summary_plot(shap_vals, X_shap, max_display=15, show=False)
plt.tight_layout(); plt.show()

### How much of the price model's |SHAP| comes from demographics?

This is one of our headline numbers. About 27% of total |SHAP| in the price model comes from the
demographic variables, even though listing-level features dominate the top of the ranking.

In [ ]:
demo_in_features = [f for f in demo_features if f in X_shap.columns]
total_abs = np.abs(shap_vals).sum()
demo_idx_list = [list(X_shap.columns).index(f) for f in demo_in_features]
demo_abs = np.abs(shap_vals[:, demo_idx_list]).sum()
demo_share = demo_abs / total_abs * 100
print(f"demographics share of total |SHAP|: {demo_share:.1f}%")

## 11. City-level analysis

Does the demographic effect on price look the same in every city, or is it driven by cross-city
variation? We refit the price model within each of the 8 largest cities and compare.

Two metrics per city:

1. **R² gain from adding demographics** — how much do tract-level features help inside this city?
2. **Standardized LR coefficient on `pct_black`** (with all controls) — the conditional effect of
   neighborhood %Black on log-price within this city.

Note: LA, NYC, San Francisco, and Portland get dropped here because they have no valid price data
in the Inside Airbnb files we downloaded — a real limitation we discuss in section 13.

In [ ]:
city_listing = [
    "accommodates", "bedrooms", "beds", "bathrooms",
    "minimum_nights", "availability_365", "availability_30",
    "number_of_reviews", "reviews_per_month",
    "review_scores_rating",
    "host_response_rate", "host_acceptance_rate",
    "has_wifi", "has_parking", "has_pool", "has_gym",
    "instant_bookable", "host_is_superhost"
]
city_demo = [
    "median_household_income", "median_gross_rent", "median_home_value",
    "poverty_rate", "pct_white", "pct_black",
    "pct_asian", "pct_hispanic", "pct_college_educated",
    "unemployment_rate"
]

room_dummies_full = pd.get_dummies(df["room_type"], prefix="room", drop_first=True)
top_cities = df["city"].value_counts().head(8).index.tolist()
print("cities:", top_cities)

results = []
for city in top_cities:
    sub = df[df["city"] == city].copy()
    sub = sub[sub["price"].notna() & (sub["price"] > 0)]
    if len(sub) < 1000:
        continue

    feats_c = pd.concat([sub[city_listing + city_demo],
                         room_dummies_full.loc[sub.index]], axis=1)
    bcols2 = feats_c.select_dtypes(include=["bool"]).columns
    feats_c[bcols2] = feats_c[bcols2].astype(int)

    imp_c = SimpleImputer(strategy="median")
    Xc = pd.DataFrame(imp_c.fit_transform(feats_c), columns=feats_c.columns, index=feats_c.index)
    yc = np.log1p(sub["price"])
    if len(Xc) < 500:
        continue

    Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.2, random_state=SEED)

    rf_full = RandomForestRegressor(n_estimators=150, max_depth=12, min_samples_leaf=10,
                                    random_state=SEED, n_jobs=-1)
    rf_full.fit(Xc_tr, yc_tr)
    r2_full = r2_score(yc_te, rf_full.predict(Xc_te))

    listing_cols_c = [c for c in Xc_tr.columns if c not in city_demo]
    rf_l_c = RandomForestRegressor(n_estimators=150, max_depth=12, min_samples_leaf=10,
                                   random_state=SEED, n_jobs=-1)
    rf_l_c.fit(Xc_tr[listing_cols_c], yc_tr)
    r2_listing_c = r2_score(yc_te, rf_l_c.predict(Xc_te[listing_cols_c]))

    sc_c = StandardScaler()
    Xs_c = sc_c.fit_transform(Xc_tr)
    lr_c = LinearRegression()
    lr_c.fit(Xs_c, yc_tr)
    pb_idx = list(Xc_tr.columns).index("pct_black")
    pct_black_coef = lr_c.coef_[pb_idx]

    results.append({
        "city": city, "n": len(Xc),
        "median_price": np.expm1(yc).median(),
        "r2_listing_only": r2_listing_c,
        "r2_with_demo": r2_full,
        "r2_gain_from_demo": r2_full - r2_listing_c,
        "pct_black_coef_lr": pct_black_coef,
    })
    print(f"  {city}: n={len(Xc)}, R2={r2_full:.3f}, "
          f"gain={r2_full-r2_listing_c:+.4f}, pct_black coef={pct_black_coef:+.4f}")

city_df = pd.DataFrame(results).sort_values("n", ascending=False)
city_df

### Figure 14 - per-city analysis

Hawaii gets the largest R² gain from adding demographics (+0.064). The within-city `pct_black`
coefficient is small in most cities (Hawaii -0.028, Nashville -0.040, San Diego -0.001, Austin -0.014),
and meaningfully positive in two (Clark/Vegas +0.084, Broward +0.081). So the strong negative
`pct_black` coefficient we see in the pooled model **comes mostly from cross-city variation**, not
from a uniform within-city pattern.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ypos = np.arange(len(city_df))

axes[0].barh(ypos, city_df["r2_gain_from_demo"], color="#20808D")
axes[0].set_yticks(ypos); axes[0].set_yticklabels(city_df["city"])
axes[0].invert_yaxis()
axes[0].set_xlabel("Gain in R² from adding demographics")
axes[0].set_title("How much demographics help price prediction (per city)")
axes[0].axvline(0, color="black", linewidth=0.5)

cols = ["#A84B2F" if c < 0 else "#20808D" for c in city_df["pct_black_coef_lr"]]
axes[1].barh(ypos, city_df["pct_black_coef_lr"], color=cols)
axes[1].set_yticks(ypos); axes[1].set_yticklabels(city_df["city"])
axes[1].invert_yaxis()
axes[1].set_xlabel("Standardized LR coefficient on pct_black\n(controlling for listing features)")
axes[1].set_title("Effect of neighborhood %Black on log price (per city)")
axes[1].axvline(0, color="black", linewidth=0.5)

plt.tight_layout(); plt.show()

## 12. Fairness analysis

We focus on the occupancy classifier here. Two protected groupings, both at the census tract level:

- **High-Black tract** = `pct_black` in top quartile.
- **Low-income tract** = `median_household_income` in bottom quartile.

We compute two fairness metrics per grouping:

- **Demographic parity** — gap in selection rate (P(predicted high-occupancy) per group).
- **Equalized odds** — gap in true positive rate, gap in false positive rate.

In [ ]:
y_pred_o_test = xgb_clf.predict(X_te_o2)
y_prob_o_test = xgb_clf.predict_proba(X_te_o2)[:, 1]

test_demos = df.loc[X_te_o2.index, ["pct_black", "pct_white", "pct_hispanic",
                                     "median_household_income", "city"]].copy()
test_demos["y_true"] = y_te_o2.values
test_demos["y_pred"] = y_pred_o_test
test_demos["y_prob"] = y_prob_o_test

thr_b = test_demos["pct_black"].quantile(0.75)
test_demos["high_black_nbhd"] = (test_demos["pct_black"] > thr_b).astype(int)

thr_i = test_demos["median_household_income"].quantile(0.25)
test_demos["low_income_nbhd"] = (test_demos["median_household_income"] < thr_i).astype(int)


def fairness_summary(grp_col):
    rows = []
    for grp_val, grp in test_demos.groupby(grp_col):
        sel = grp["y_pred"].mean()
        tpr = grp.loc[grp["y_true"] == 1, "y_pred"].mean() if (grp["y_true"] == 1).any() else np.nan
        fpr = grp.loc[grp["y_true"] == 0, "y_pred"].mean() if (grp["y_true"] == 0).any() else np.nan
        rows.append({"group": f"{grp_col}={grp_val}", "n": len(grp),
                     "selection_rate": sel, "TPR": tpr, "FPR": fpr,
                     "true_pos_rate": grp["y_true"].mean()})
    return pd.DataFrame(rows)

def gap(d, grp_col, metric):
    a = d.loc[d["group"] == f"{grp_col}=1", metric].iloc[0]
    b = d.loc[d["group"] == f"{grp_col}=0", metric].iloc[0]
    return abs(a - b)

fb = fairness_summary("high_black_nbhd")
print("by neighborhood race composition:")
print(fb.to_string(index=False))
dp_b = gap(fb, "high_black_nbhd", "selection_rate")
tpr_b = gap(fb, "high_black_nbhd", "TPR")
fpr_b = gap(fb, "high_black_nbhd", "FPR")
print(f"DP gap = {dp_b:.4f}, TPR gap = {tpr_b:.4f}, FPR gap = {fpr_b:.4f}")

print("\nby neighborhood income:")
fi = fairness_summary("low_income_nbhd")
print(fi.to_string(index=False))
dp_i = gap(fi, "low_income_nbhd", "selection_rate")
tpr_i = gap(fi, "low_income_nbhd", "TPR")
fpr_i = gap(fi, "low_income_nbhd", "FPR")
print(f"DP gap = {dp_i:.4f}, TPR gap = {tpr_i:.4f}, FPR gap = {fpr_i:.4f}")

### Figure 13 - fairness metrics

Demographic parity gaps are small (under ~2 percentage points). Listings in high-Black tracts are
slightly more likely to be flagged as high-occupancy (52.2% vs 50.4%) but the actual base rate is
also higher there, so most of the gap is base-rate, not model bias. Same story for low-income tracts.

We want to be careful reading this. Small parity gaps **do not mean the system is fair** in some
final sense — the protected groupings are tract-level rather than host-level, and the small parity
gaps for occupancy coexist with a substantial demographic effect on **price** (a different kind of
disparity). So the fair-looking occupancy classifier sits inside a system where price already
encodes demographic structure.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
metrics = {
    "Selection rate": [fb.loc[fb["group"] == "high_black_nbhd=0", "selection_rate"].iloc[0],
                       fb.loc[fb["group"] == "high_black_nbhd=1", "selection_rate"].iloc[0]],
    "TPR": [fb.loc[fb["group"] == "high_black_nbhd=0", "TPR"].iloc[0],
            fb.loc[fb["group"] == "high_black_nbhd=1", "TPR"].iloc[0]],
    "FPR": [fb.loc[fb["group"] == "high_black_nbhd=0", "FPR"].iloc[0],
            fb.loc[fb["group"] == "high_black_nbhd=1", "FPR"].iloc[0]],
}
x = np.arange(len(metrics)); w = 0.35
ax.bar(x - w/2, [v[0] for v in metrics.values()], w, label="Other tracts", color="#95a5a6")
ax.bar(x + w/2, [v[1] for v in metrics.values()], w, label="High-Black tracts", color="#A84B2F")
ax.set_xticks(x); ax.set_xticklabels(list(metrics.keys()))
ax.set_ylabel("Rate"); ax.set_title("Fairness by neighborhood race composition")
ax.legend(); ax.set_ylim(0, 1)

ax = axes[1]
metrics = {
    "Selection rate": [fi.loc[fi["group"] == "low_income_nbhd=0", "selection_rate"].iloc[0],
                       fi.loc[fi["group"] == "low_income_nbhd=1", "selection_rate"].iloc[0]],
    "TPR": [fi.loc[fi["group"] == "low_income_nbhd=0", "TPR"].iloc[0],
            fi.loc[fi["group"] == "low_income_nbhd=1", "TPR"].iloc[0]],
    "FPR": [fi.loc[fi["group"] == "low_income_nbhd=0", "FPR"].iloc[0],
            fi.loc[fi["group"] == "low_income_nbhd=1", "FPR"].iloc[0]],
}
ax.bar(x - w/2, [v[0] for v in metrics.values()], w, label="Higher-income", color="#95a5a6")
ax.bar(x + w/2, [v[1] for v in metrics.values()], w, label="Low-income tracts", color="#1B474D")
ax.set_xticks(x); ax.set_xticklabels(list(metrics.keys()))
ax.set_ylabel("Rate"); ax.set_title("Fairness by neighborhood income")
ax.legend(); ax.set_ylim(0, 1)

plt.tight_layout(); plt.show()

## 13. Things that did not work

A few honest notes about what we tried and what failed.

**Linear regression on price.** Even after the log transform, RMSE in dollar space was ~$275 because
the model overestimated on the long tail of high-end listings. We had hoped log would mostly fix
this; tree models handled it much better because they are not sensitive to scale on the tails.

**Demographics for occupancy.** We expected demographics to help. The ablation showed they don't
(ΔAUC ≈ 0, slightly negative for RF). This held across RF, XGBoost, LightGBM, and 5-fold CV folds.
Our current interpretation is that occupancy is dominated by listing-level signals
(`reviews_per_month`, `number_of_reviews`, host responsiveness) that already absorb whatever
demographic signal exists.

**KNN imputation.** Slower *and* worse than median. The high-dimensional mixed-scale feature space
hurts the distance metric. We ended up using median imputation as a more honest default.

**FBI Crime Data Explorer features.** We could not align them at the right resolution — the most
granular CDE data is county-level, which is much coarser than our census-tract level demographics.

## 14. Discussion and findings

Returning to our research questions:

1. **Yes, ML models predict occupancy well** (AUC 0.986, XGBoost) and **predict price reasonably**
   (R² 0.807). The features that drive each are very different: listing size and capacity dominate
   price; review counts and host responsiveness dominate occupancy.

2. **Listing characteristics for price.** `accommodates`, `bedrooms`, `bathrooms`, `prop_private_room`,
   `minimum_nights`, `reviews_per_month` are the strongest listing-side signals.

3. **Demographics matter for price, not occupancy.** Demographics account for ~27% of total |SHAP| in
   the price model and improve out-of-sample R² by +0.074, but contribute essentially nothing once
   listing signals are in the occupancy model.

The policy-relevant finding is that **the share of Black residents in a census tract is a meaningful
negative predictor of price** even after controlling for listing size, amenities, review scores,
host attributes, and other neighborhood characteristics like income, rent, and home value.

A standardized linear coefficient of -0.213 in the pooled model translates to roughly a **19% lower
predicted price** for listings in tracts that are one standard deviation higher in `pct_black`. SHAP
confirms this in the XGBoost model: high `pct_black` values systematically push predictions down.

We want to be honest that the per-city analysis (section 11) shows the within-city `pct_black`
coefficient is small in most individual cities, which means the strong pooled effect is partly a
**cross-city pattern** (cities with higher Black shares tend to have lower average prices) rather
than a uniform within-neighborhood pattern. This connects to the findings of Edelman, Luca, and
Svirsky (2017) and Tornberg (2022) — platforms like Airbnb appear to encode racial structure in
their pricing equilibrium, but the geography of that effect is not uniform.

A second contribution is the contrast between price and occupancy. The fairness metrics for the
occupancy classifier are only marginally different (parity gaps under 2.5 percentage points) — that
looks fair. But it sits next to a price model that does encode demographic structure. So disparities
in this ecosystem may operate primarily through **price formation**, not through demand prediction.
That is a useful distinction for thinking about where interventions could go.

## 15. Limitations

1. **Missing price data.** 43% of listings have no price, including all listings from LA, NYC,
   San Francisco, and Portland (~33% of the dataset combined). Imputation results are stable but
   our price model is in practice a price model for the 34 cities minus those four. A robust
   replication would need price data from a different source.

2. **Cross-sectional setup.** 2025 listings + 2023 ACS demographics = no causal claims. The negative
   `pct_black` coefficient is a conditional association. The mechanism could be guest demand-side
   preferences, host pricing decisions, platform recommendation effects, or unobserved amenities.
   We are not claiming this identifies discrimination — we are claiming it documents a disparity
   that aligns with prior experimental and observational work.

3. **Tract-level proxies, not host-level.** Tornberg (2022) showed host-level racial revenue gaps
   exist *within* similar neighborhoods. Our setup cannot detect that pattern.

4. **Occupancy has leakage.** `reviews_per_month` and `number_of_reviews` are themselves caused by
   occupancy, so we are partially predicting recent occupancy from lagged occupancy proxies. AUC
   0.986 is high in part because of this. A cleaner version would use only ex-ante features.

## 16. Future work

- Recover price data for LA, NYC, San Francisco, and Portland from a different source or directly
  from Airbnb's API, and rerun the city-level analysis.
- Add the FBI Crime Data Explorer features at a properly aligned geographic resolution.
- Replace post-hoc occupancy proxies (review-derived features) with ex-ante features only.
- Try counterfactual fairness: perturb `pct_black` holding everything else fixed, to estimate
  a more causal demographic effect.
- Explore host-level race signals (with appropriate ethical guardrails) to compare with the host-level
  disparity in Tornberg (2022).

## 17. References

- Edelman, B., Luca, M., & Svirsky, D. (2017). *Racial Discrimination in the Sharing Economy:
  Evidence from a Field Experiment.* American Economic Journal: Applied Economics, 9(2), 1-22.
- Tornberg, P. (2022). *How sharing is the "sharing economy"? Evidence from 97 Airbnb markets.*
  PLOS ONE 17(4): e0266998.
- Lundberg, S. M., & Lee, S.-I. (2017). *A Unified Approach to Interpreting Model Predictions.*
  NeurIPS 30.
- Inside Airbnb (2025). *Get the Data.* insideairbnb.com/get-the-data
- US Census Bureau (2023). *American Community Survey 5-Year Estimates.*